In [1]:
DATA = '../data/'
FIGS = '../results/figures/'
CACHE = '../f1_cache'

In [2]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_pickle(DATA + 'laps_clean.pkl') # load the clean lap data

# Hold out the demo race entirely
demo = df[(df['Year'] == 2025) & (df['Circuit'] == 'Barcelona')]
assert len(demo) > 0, "demo filter matched nothing - check df['Circuit'].unique()"
pool = df.drop(demo.index)
print(f"demo race: {len(demo)} laps, training pool: {len(pool)} laps")

# Split BY RACE WEEKEND, never randomly by lap
races = pool['RaceID'].unique()
rng = np.random.default_rng(0)
test_races = rng.choice(races, size=int(len(races) * 0.2), replace=False) # randomly selected 20% of races for testing

train = pool[~pool['RaceID'].isin(test_races)] # the pile of laps used for training the model
test = pool[pool['RaceID'].isin(test_races)] # the pile of laps used for scoring the model's accuracy
print(f"train: {len(train)} laps / test: {len(test)} laps")

# --- Tier 1: heuristic ---
ref = (train.groupby(['Circuit', 'Driver'])['LapSeconds']
            .median().rename('pred1').reset_index())
t1 = test.merge(ref, on=['Circuit', 'Driver'], how='inner')
mae1 = mean_absolute_error(t1['LapSeconds'], t1['pred1'])

# The inner join drops test laps whose (Circuit, Driver) pair never appeared in training.
# Tier 2 must be scored on the SAME laps or the comparison is meaningless.
keep = test.set_index(['Circuit', 'Driver']).index.isin(
    ref.set_index(['Circuit', 'Driver']).index)
print(f"tier 1 scored on {keep.sum()} of {len(test)} test laps")

# --- Tier 2: linear regression ---
feat = ['TyreLife', 'LapNumber', 'AirTemp', 'TrackTemp'] # the continuous numerical values that set the movement (produces the slope)
cats = ['Compound', 'Circuit', 'Driver', 'Team'] # the categorical features which set where the lap time sits (a fixed offset for lap times)

Xtr = pd.get_dummies(train[feat + cats], columns=cats, drop_first=True) # builds the training input table
Xte = pd.get_dummies(test[feat + cats], columns=cats, drop_first=True) # builds the test input table
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0) # force the test set's column structure to match training set

lin = LinearRegression().fit(Xtr, train['LapSeconds'])
pred2 = lin.predict(Xte)
mae2 = mean_absolute_error(test['LapSeconds'], pred2) # on all test laps
mae2_fair = mean_absolute_error(test['LapSeconds'][keep], pred2[keep]) # on tier 1's laps

print(f"\nTier 1 (heuristic): {mae1:.3f} s")
print(f"Tier 2 (linear):    {mae2_fair:.3f} s  <- same laps as tier 1, this is the comparison")
print(f"improvement:        {mae1 - mae2_fair:.3f} s")
print(f"Tier 2, all laps:   {mae2:.3f} s  <- use this one against tier 3")

coef = dict(zip(Xtr.columns, lin.coef_))
print(f"\nTyreLife coefficient:  {coef['TyreLife']:+.4f} s per lap of age")
print(f"LapNumber coefficient: {coef['LapNumber']:+.4f} s per lap of race")

demo race: 988 laps, training pool: 78394 laps
train: 63104 laps / test: 15290 laps
tier 1 scored on 12800 of 15290 test laps

Tier 1 (heuristic): 1.674 s
Tier 2 (linear):    1.170 s  <- same laps as tier 1, this is the comparison
improvement:        0.504 s
Tier 2, all laps:   1.284 s  <- use this one against tier 3

TyreLife coefficient:  +0.0256 s per lap of age
LapNumber coefficient: -0.0464 s per lap of race


/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
